In [0]:
from pyspark.sql import SparkSession, functions as F
import json, uuid, requests, os
from datetime import datetime
from databricks.sdk import WorkspaceClient

spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

# 1. Aggregate symptom frequencies across all records
symptom_df = spark.sql("""
    WITH exploded_symptoms AS (
        SELECT explode(symptoms) as symptom, hospital_id
        FROM workspace.vaidya.patient_records
        WHERE timestamp >= current_timestamp() - interval 7 days
    )
    SELECT symptom, 
           count(*) as frequency,
           collect_set(hospital_id) as hospitals
    FROM exploded_symptoms
    GROUP BY symptom
    ORDER BY frequency DESC
    LIMIT 20
""")
symptom_df.show()

# 2. Use Llama to generate human-readable insight
top_symptoms = [(r["symptom"], r["frequency"]) for r in symptom_df.collect()]
prompt = f"""You are a public health AI for India.
Given the top symptoms seen across hospitals this week:
{json.dumps(top_symptoms)}

Generate 3 concise alert bullets for doctors. Each should be:
- Actionable (what to watch for)
- Evidence-based (based on the symptom data above)
- India-specific (mention seasonal context if relevant for March)

Respond as JSON: {{"alerts": ["...", "...", "..."]}}"""

# Reuse the same AI Gateway pattern
from databricks.sdk.service.serving import ChatMessage
headers = w.config.authenticate()
resp = requests.post(
    f"{os.environ.get('LLM_OPENAI_BASE_URL')}/chat/completions",
    headers={**headers, "Content-Type": "application/json"},
    json={
        "model": "databricks-llama-4-maverick",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 512, "temperature": 0.3
    }
)
content = resp.json()["choices"][0]["message"]["content"]
content = content.strip().lstrip("```json").rstrip("```").strip()
alerts_data = json.loads(content)

# 3. Write to health_alerts Delta table
from pyspark.sql import Row
rows = [Row(
    alert_id=str(uuid.uuid4()),
    generated_at=datetime.now(),
    alert_type="symptom_trend",
    insight_text=alert,
    top_symptoms=[s[0] for s in top_symptoms[:5]],
    affected_hospitals=list({h for r in symptom_df.collect() for h in r["hospitals"]}),
    severity="INFO"
) for alert in alerts_data["alerts"]]

alert_df = spark.createDataFrame(rows)
alert_df.write.mode("append").saveAsTable("workspace.vaidya.health_alerts")
print("Alerts written:", len(rows))

In [0]:
!pip install databricks-sql-connector

In [0]:
dbutils.library.restartPython()

In [0]:
# Run this on your cluster to populate realistic records
import uuid, json, random
from datetime import datetime, timedelta
from databricks import sql
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.replace("https://", "")
warehouses = list(w.warehouses.list())
wh_id = warehouses[0].id
token = w.config.authenticate().get("Authorization","").replace("Bearer ","")

DOCTORS = ["DR001", "DR002", "DR003", "DR004"]

CASES = [
    {"symptoms": ["fever", "chills", "body ache"], "diagnosis": "Viral fever", "medications": ["Paracetamol 500mg TDS", "ORS"], "soap_s": "Patient complaints of fever since 3 days with chills and body ache.", "soap_o": "Temp 101.2F, HR 98, BP 110/70", "soap_a": "Acute viral fever", "soap_p": "Paracetamol 500mg TDS x 5 days, rest, fluids, review if not improving"},
    {"symptoms": ["cough", "sore throat", "runny nose"], "diagnosis": "Upper respiratory tract infection", "medications": ["Cetirizine 10mg OD", "Cough syrup 10ml TDS"], "soap_s": "Cough and cold for 4 days, sore throat, mild fever.", "soap_o": "Throat congested, mild lymphadenopathy", "soap_a": "URTI", "soap_p": "Symptomatic treatment, steam inhalation, warm fluids"},
    {"symptoms": ["chest pain", "shortness of breath"], "diagnosis": "Acute coronary syndrome - rule out", "medications": ["Aspirin 325mg", "Nitroglycerine SL"], "soap_s": "Chest pain since morning, radiating to left arm, sweating.", "soap_o": "BP 150/90, HR 102, ECG changes noted", "soap_a": "ACS rule out, unstable angina", "soap_p": "ECG, troponin, echo, cardiology referral"},
    {"symptoms": ["knee pain", "swelling"], "diagnosis": "Osteoarthritis knee", "medications": ["Diclofenac 50mg BD", "Pantoprazole 40mg OD"], "soap_s": "Right knee pain and swelling for 2 weeks, worse on climbing stairs.", "soap_o": "Swelling +, tenderness +, crepitus +, ROM restricted", "soap_a": "OA right knee grade 2", "soap_p": "NSAIDs, physiotherapy referral, weight reduction advised"},
    {"symptoms": ["headache", "vomiting", "photophobia"], "diagnosis": "Migraine", "medications": ["Sumatriptan 50mg", "Domperidone 10mg"], "soap_s": "Severe headache with vomiting, sensitivity to light.", "soap_o": "Alert, neck supple, no neuro deficits", "soap_a": "Migraine with aura", "soap_p": "Triptan, dark room rest, prophylaxis discussed"},
    {"symptoms": ["abdominal pain", "loose stools", "nausea"], "diagnosis": "Acute gastroenteritis", "medications": ["ORS", "Ondansetron 4mg", "Metronidazole 400mg TDS"], "soap_s": "Loose stools 6 times since last night, nausea and cramps.", "soap_o": "Mild dehydration, tenderness in periumbilical region", "soap_a": "AGE with mild dehydration", "soap_p": "ORS, bland diet, Metronidazole 5 days, review if worsening"},
    {"symptoms": ["burning urination", "frequency"], "diagnosis": "Urinary tract infection", "medications": ["Nitrofurantoin 100mg BD", "Phenazopyridine"], "soap_s": "Burning sensation while urinating, increased frequency for 2 days.", "soap_o": "Urine routine: pus cells 20-25/hpf, bacteria +", "soap_a": "Lower UTI", "soap_p": "Nitrofurantoin 7 days, increase fluid intake, urine culture sent"},
    {"symptoms": ["rash", "itching", "hives"], "diagnosis": "Urticaria", "medications": ["Levocetrizine 5mg OD", "Methylprednisolone 8mg"], "soap_s": "Sudden onset rash all over body after eating seafood.", "soap_o": "Urticarial wheals on trunk and extremities, no angioedema", "soap_a": "Allergic urticaria - food trigger", "soap_p": "Antihistamines, steroids, avoid seafood, EpiPen prescribed"},
    {"symptoms": ["back pain", "leg pain"], "diagnosis": "Lumbar disc prolapse", "medications": ["Etoricoxib 90mg OD", "Pregabalin 75mg BD"], "soap_s": "Low back pain radiating to right leg for 1 month, worse on bending.", "soap_o": "SLR positive at 40 degrees right, mild motor weakness", "soap_a": "L4-L5 disc prolapse with radiculopathy", "soap_p": "MRI lumbar spine, physio, pain management, neurosurgery if no improvement"},
    {"symptoms": ["fatigue", "pale", "breathlessness on exertion"], "diagnosis": "Iron deficiency anaemia", "medications": ["Ferrous sulphate 200mg BD", "Vitamin C 500mg"], "soap_s": "Tiredness for months, breathless on walking, looks pale.", "soap_o": "Pallor ++, HR 104, Hb 7.2 g/dL", "soap_a": "Severe iron deficiency anaemia", "soap_p": "Iron supplementation 3 months, dietary advice, recheck Hb after 6 weeks"},
]

LANGUAGES = ["en-IN", "hi-IN", "mr-IN", "en-IN", "hi-IN"]
PATIENTS = [f"PAT{1000+i}" for i in range(40)]

rows = []
now = datetime.now()

for i in range(60):
    case = random.choice(CASES)
    doctor = random.choice(DOCTORS)
    # Spread over last 7 days, weighted toward today
    days_ago = random.choices([0,0,0,1,1,2,3,4,5,6], k=1)[0]
    hours_ago = random.randint(0, 8)
    ts = now - timedelta(days=days_ago, hours=hours_ago)
    lang = random.choice(LANGUAGES)

    rows.append({
        "record_id": str(uuid.uuid4()),
        "patient_id": random.choice(PATIENTS),
        "doctor_id": doctor,
        "hospital_id": "DEMO_HOSPITAL",
        "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "language_detected": lang,
        "raw_transcript": case["soap_s"],
        "structured_note": json.dumps(case),
        "soap_subjective": case["soap_s"],
        "soap_objective": case["soap_o"],
        "soap_assessment": case["soap_a"],
        "soap_plan": case["soap_p"],
        "is_anonymized": False,
    })

# Insert via SQL
with sql.connect(server_hostname=host, http_path=f"/sql/1.0/warehouses/{wh_id}", access_token=token) as conn:
    with conn.cursor() as cur:
        for r in rows:
            def esc(s): return str(s or "").replace("'","''")
            cur.execute(f"""
                INSERT INTO workspace.vaidya.patient_records
                (record_id, patient_id, doctor_id, hospital_id, timestamp,
                 language_detected, raw_transcript, structured_note,
                 soap_subjective, soap_objective, soap_assessment, soap_plan, is_anonymized)
                VALUES ('{r["record_id"]}','{r["patient_id"]}','{r["doctor_id"]}',
                '{r["hospital_id"]}','{r["timestamp"]}','{r["language_detected"]}',
                '{esc(r["raw_transcript"])}','{esc(r["structured_note"])}',
                '{esc(r["soap_subjective"])}','{esc(r["soap_objective"])}',
                '{esc(r["soap_assessment"])}','{esc(r["soap_plan"])}',false)
            """)

print(f"Inserted {len(rows)} records")